# 家計調査データ取得ノートブック(e-Stat API)

**目的**: 「あすけん」のような家計簿アプリで、ユーザーの支出傾向を総務省統計局「家計調査」の統計データと比較し、
「貯蓄できている人の特徴」を分析するための元データを取得する。

参考になりそうな統計:
- **貯蓄・負債編**: 年齢階級別・年間収入階級別の貯蓄現在高・負債現在高
- **家計収支編**: 実収入・実支出・黒字(貯蓄純増)・黒字率・平均消費性向

## 事前準備: appId(アプリケーションID)の取得

1. https://www.e-stat.go.jp/mypage/user/preregister にアクセスしてユーザー登録(無料・メール認証あり)
2. ログイン後、マイページ > 「アプリケーションID」から新規発行
3. 発行されたIDを下のセルで環境変数 `ESTAT_APP_ID` にセットする(コードに直書きしない)

```bash
# ターミナルで(このノートブックを開く前 or 別セルで)
export ESTAT_APP_ID="あなたのアプリケーションID"
```

もしくは下の `os.environ` のセルに直接一時的にセットしてもOK(ただしGit管理下に置かない・共有しないこと)。

In [ ]:
import os
import requests
import pandas as pd

# 環境変数から読む(推奨)。未設定ならここで一時的にセットしてもよい。
APP_ID = os.environ.get("ESTAT_APP_ID", "")

if not APP_ID:
    raise RuntimeError(
        "ESTAT_APP_ID が未設定です。上のMarkdownセルの手順でappIdを取得し、\n"
        "export ESTAT_APP_ID=... してから Jupyter を起動するか、\n"
        "このセルで APP_ID = 'xxxx' のように一時的に設定してください。"
    )

BASE_URL = "https://api.e-stat.go.jp/rest/3.0/app/json"
KAKEI_GOV_STATS_CODE = "00200561"  # 家計調査の政府統計コード

## 1. 家計調査の統計表一覧を検索する

`getStatsList` で政府統計コード(家計調査 = `00200561`)配下の統計表を検索し、
`searchWord` で絞り込む。まずは候補を一覧表示して、目的に合う `statsDataId` を選ぶ。

In [ ]:
def search_stats_list(search_word: str, limit: int = 30) -> pd.DataFrame:
    """家計調査配下の統計表をキーワード検索し、DataFrameで返す"""
    params = {
        "appId": APP_ID,
        "statsCode": KAKEI_GOV_STATS_CODE,
        "searchWord": search_word,
        "limit": limit,
    }
    res = requests.get(f"{BASE_URL}/getStatsList", params=params, timeout=30)
    res.raise_for_status()
    body = res.json()["GET_STATS_LIST"]

    result = body["RESULT"]
    if result["STATUS"] != 0:
        raise RuntimeError(f"e-Stat APIエラー: {result.get('ERROR_MSG')}")

    table_inf = body["DATALIST_INF"].get("TABLE_INF", [])
    if isinstance(table_inf, dict):  # 結果が1件だとdictで返ってくる
        table_inf = [table_inf]

    rows = [
        {
            "statsDataId": t["@id"],
            "title": t["TITLE"].get("$", t["TITLE"]) if isinstance(t["TITLE"], dict) else t["TITLE"],
            "survey_date": t.get("SURVEY_DATE"),
            "updated": t.get("UPDATED_DATE"),
        }
        for t in table_inf
    ]
    return pd.DataFrame(rows)


# 例: 貯蓄・負債に関する統計表を検索
search_stats_list("貯蓄")

In [ ]:
# 例: 黒字率・平均消費性向など、家計収支バランスに関する統計表を検索
search_stats_list("黒字率")

## 2. 統計表のメタ情報を確認する

上の一覧から気になる `statsDataId` を選んだら、`getMetaInfo` でその表がどんな分類軸
(年齢階級・年間収入階級・世帯区分など)を持っているかを確認する。

In [ ]:
def get_meta_info(stats_data_id: str) -> pd.DataFrame:
    """統計表の分類軸(CLASS_OBJ)一覧を取得する"""
    params = {"appId": APP_ID, "statsDataId": stats_data_id}
    res = requests.get(f"{BASE_URL}/getMetaInfo", params=params, timeout=30)
    res.raise_for_status()
    body = res.json()["GET_META_INFO"]

    if body["RESULT"]["STATUS"] != 0:
        raise RuntimeError(f"e-Stat APIエラー: {body['RESULT'].get('ERROR_MSG')}")

    class_obj = body["METADATA_INF"]["CLASS_INF"]["CLASS_OBJ"]
    rows = [{"axis_id": c["@id"], "axis_name": c["@name"]} for c in class_obj]
    return pd.DataFrame(rows)


# STATS_DATA_ID = "0003xxxxxx"  # 上の検索結果から選んだIDに置き換える
# get_meta_info(STATS_DATA_ID)

## 3. 実データを取得する

`statsDataId` が決まったら `getStatsData` で実データを取得し、DataFrame化する。
分類軸で絞り込みたい場合は `cdCat01`, `cdArea` などのパラメータを追加する
(具体的な軸コードは `get_meta_info` の結果から確認)。

In [ ]:
def get_stats_data(stats_data_id: str, **extra_params) -> pd.DataFrame:
    """統計データを取得しDataFrameで返す"""
    params = {"appId": APP_ID, "statsDataId": stats_data_id, **extra_params}
    res = requests.get(f"{BASE_URL}/getStatsData", params=params, timeout=30)
    res.raise_for_status()
    body = res.json()["GET_STATS_DATA"]

    if body["RESULT"]["STATUS"] != 0:
        raise RuntimeError(f"e-Stat APIエラー: {body['RESULT'].get('ERROR_MSG')}")

    values = body["STATISTICAL_DATA"]["DATA_INF"]["VALUE"]
    return pd.DataFrame(values)


# 例:
# df = get_stats_data(STATS_DATA_ID, limit=1000)
# df.head()

## 4. 可視化(例)

取得したデータを年齢階級別・年収階級別などで比較し、貯蓄できている世帯の傾向を見る。
実際の列名はメタ情報(`get_meta_info`)で確認したコード(`cat01`, `area` など)に置き換えること。

In [ ]:
import matplotlib.pyplot as plt

# 例: 取得したdfに 'cat01' (年齢階級コード) と '$' (値) 列がある想定
# pivot = df.pivot_table(index="cat01", values="$", aggfunc="mean")
# pivot.plot(kind="bar", title="年齢階級別 平均値")
# plt.show()